# Centralized Analysis for fl-tabular

This notebook is a centralized baseline for comparison with the Flower run. It uses the exact train, validation, and test datasets, drops `CRF01` from the analysis, trains one model end to end, prints progress during training, saves the final model, and evaluates the saved model on the held-out test split.

In [ ]:
# ----------------------------- Import Libraries ----------------------------- #
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from fltabular.task import CostRegressor, IGNORED_COLUMNS, _build_preprocessor, evaluator, get_input_dim
# -------------------------- Define Hyperparameters -------------------------- #
BATCH_SIZE = 16
NUM_EPOCHS = 800
LEARNING_RATE = 0.5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# --------------------------------- Load Data -------------------------------- #
def find_file_upwards(filename: str) -> Path:
    for directory in [Path.cwd(), *Path.cwd().parents]:
        candidate = directory / filename
        if candidate.exists():
            return candidate.resolve()
    raise FileNotFoundError(f'Could not find {filename} in {Path.cwd()} or any parent directory.')
train_path = find_file_upwards('train_db.pkl')
val_path = find_file_upwards('val_db.pkl')
test_path = find_file_upwards('test_db.pkl')
# print(f'Using device: {DEVICE}')
# print(f'Train data: {train_path}')
# print(f'Val data:   {val_path}')
# print(f'Test data:  {test_path}')

Using device: cpu
Train data: /home/ti-interns-1/Internship-2/flower/fl-tabular/train_db.pkl
Val data:   /home/ti-interns-1/Internship-2/flower/fl-tabular/val_db.pkl
Test data:  /home/ti-interns-1/Internship-2/flower/fl-tabular/test_db.pkl


In [ ]:
# ---------------------------- Data Preprosessing ---------------------------- #
def _prepare_frame(dataset: pd.DataFrame):
    dataset = dataset.dropna().reset_index(drop=True)
    target_column = 'COST_BL'
    feature_frame = dataset.drop(columns=[target_column], errors='ignore')
    feature_frame = feature_frame.drop(columns=list(IGNORED_COLUMNS), errors='ignore')
    y = pd.to_numeric(dataset[target_column], errors='coerce')
    valid_rows = y.notna()
    feature_frame = feature_frame.loc[valid_rows].reset_index(drop=True)
    y = y.loc[valid_rows].reset_index(drop=True)
    if feature_frame.empty:
        raise ValueError('A dataset split has no usable rows after target filtering.')
    return feature_frame, y


In [ ]:
# ----------------------- Data Loaders and Preprocessor ---------------------- #
def load_exact_splits(batch_size: int = BATCH_SIZE):
    train_dataset = pd.read_pickle(train_path)
    val_dataset = pd.read_pickle(val_path)
    test_dataset = pd.read_pickle(test_path)

    x_train_frame, y_train = _prepare_frame(train_dataset)
    x_val_frame, y_val = _prepare_frame(val_dataset)
    x_test_frame, y_test = _prepare_frame(test_dataset)

    preprocessor = _build_preprocessor(x_train_frame)
    x_train = preprocessor.fit_transform(x_train_frame)
    x_val = preprocessor.transform(x_val_frame)
    x_test = preprocessor.transform(x_test_frame)

    train_loader = DataLoader(
        TensorDataset(
            torch.tensor(x_train, dtype=torch.float32),
            torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1),
        ),
        batch_size=batch_size,
        shuffle=True,
    )
    val_loader = DataLoader(
        TensorDataset(
            torch.tensor(x_val, dtype=torch.float32),
            torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1),
        ),
        batch_size=batch_size,
        shuffle=False,
    )
    test_loader = DataLoader(
        TensorDataset(
            torch.tensor(x_test, dtype=torch.float32),
            torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1),
        ),
        batch_size=batch_size,
        shuffle=False,
    )
    return train_loader, val_loader, test_loader, preprocessor

In [ ]:
# ------------------------------- Model Define ------------------------------- #
def train_centralized(model, train_loader, val_loader, 
                      num_epochs: int = NUM_EPOCHS, 
                      learning_rate: float = LEARNING_RATE):
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    history = []
    model.train()
    for epoch in range(1, num_epochs + 1):
        running_loss = 0.0
        sample_count = 0
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            batch_size = y_batch.size(0)
            running_loss += loss.item() * batch_size
            sample_count += batch_size

        train_loss = running_loss / sample_count
        train_mae, train_mse, train_rmse, train_r2 = evaluator(model, train_loader)
        val_mae, val_mse, val_rmse, val_r2 = evaluator(model, val_loader)
        history.append((train_loss, train_mae, train_mse, train_rmse, 
                        train_r2, val_mae, val_mse, val_rmse, val_r2))
        print(
            f'Epoch {epoch:03d}/{num_epochs} | '
            f'train_loss={train_loss:.4f} | '
            f'train_r2={train_r2:.4f} | '
            f'val_mae={val_mae:.4f} | '
            f'val_rmse={val_rmse:.4f} | '
            f'val_r2={val_r2:.4f}'
        )

    return history

In [ ]:
# -------------- Size of the dataset splits and input dimension: ------------- #
train_loader, val_loader, test_loader, preprocessor = load_exact_splits(batch_size=BATCH_SIZE)

print(f'Train samples: {len(train_loader.dataset)}')
print(f'Validation samples: {len(val_loader.dataset)}')
print(f'Test samples: {len(test_loader.dataset)}')
print(f'Input dimension: {get_input_dim()}')

Train samples: 256
Validation samples: 64
Test samples: 81
Input dimension: 15


In [ ]:
# -------------- Feature inspection of the fitted preprocessor: -------------- #
try:
    original_features = list(preprocessor.feature_names_in_)
except AttributeError:
    original_features = []
    for name, trans, cols in preprocessor.transformers_:
        if cols == 'remainder':
            continue
        original_features.extend(list(cols))

print("Original feature columns:", original_features)

# group columns by transformer name
cat_cols, num_cols = [], []
for name, trans, cols in preprocessor.transformers_:
    if cols == 'remainder':
        continue
    cols_list = list(cols)
    if name.lower().startswith('cat') or 'Ordinal' in trans.__class__.__name__:
        cat_cols.extend(cols_list)
    else:
        num_cols.extend(cols_list)

print("Categorical columns:", cat_cols)
print("Numeric columns:", num_cols)

# show categories for ordinal encoders (if present)
for name, trans, cols in preprocessor.transformers_:
    if hasattr(trans, "categories_"):
        for col_name, cats in zip(cols, trans.categories_):
            print(f"Categories for {col_name}: {list(cats)[:50]}")  # limit output

# transformed feature names (sklearn >=1.0)
try:
    tf_names = preprocessor.get_feature_names_out()
    print("Transformed feature names:", list(tf_names))
except Exception:
    passmodel = CostRegressor(get_input_dim()).to(DEVICE)

Original feature columns: ['CRF05', 'CRF07', 'CRF10', 'CRF11', 'CRF12', 'CRF13', 'CRF14', 'CRF15', 'CRF17A', 'CRF17B', 'CRF17C', 'CRF18', 'CRF19', 'CRF57', 'V106']
Categorical columns: ['CRF05', 'CRF10', 'CRF11', 'CRF12', 'CRF13', 'CRF14', 'CRF15', 'CRF17A', 'CRF17B', 'CRF17C', 'CRF18', 'CRF19', 'CRF57']
Numeric columns: ['CRF07', 'V106']
Categories for CRF05: [np.int64(0), np.int64(1)]
Categories for CRF10: [np.int64(0), np.int64(1)]
Categories for CRF11: [np.int64(0), np.int64(1)]
Categories for CRF12: [np.int64(0), np.int64(1)]
Categories for CRF13: [np.int64(0), np.int64(1)]
Categories for CRF14: [np.int64(0), np.int64(1)]
Categories for CRF15: [np.int64(0), np.int64(1)]
Categories for CRF17A: [np.int64(0), np.int64(1)]
Categories for CRF17B: [np.int64(0), np.int64(1)]
Categories for CRF17C: [np.int64(0), np.int64(1)]
Categories for CRF18: [np.int64(0), np.int64(1)]
Categories for CRF19: [np.int64(0), np.int64(1)]
Categories for CRF57: [np.int64(0), np.int64(1)]
Transformed feature

In [ ]:
# ------------------------------ Train the model ----------------------------- #
history = train_centralized(model, train_loader, val_loader, 
                            num_epochs=NUM_EPOCHS, 
                            learning_rate=LEARNING_RATE)

model_path = Path.cwd() / 'centralized_model.pt'
torch.save(model.state_dict(), model_path)

print('\nCentralized training complete')
print(f'Saved model to: {model_path}')
print(f'Final train MAE: {history[-1][1]:.4f}')
print(f'Final train MSE: {history[-1][2]:.4f}')
print(f'Final train RMSE: {history[-1][3]:.4f}')
print(f'Final train R2: {history[-1][4]:.4f}')
print(f'Final validation MAE:  {history[-1][5]:.4f}')
print(f'Final validation MSE:  {history[-1][6]:.4f}')
print(f'Final validation RMSE: {history[-1][7]:.4f}')
print(f'Final validation R2:   {history[-1][8]:.4f}')

Epoch 001/800 | train_loss=624.4855 | train_r2=0.3911 | val_mae=746.9327 | val_rmse=2951.5043 | val_r2=0.3895
Epoch 002/800 | train_loss=624.3177 | train_r2=0.3907 | val_mae=746.9043 | val_rmse=2952.6117 | val_r2=0.3890
Epoch 003/800 | train_loss=624.2812 | train_r2=0.3908 | val_mae=746.9088 | val_rmse=2952.4175 | val_r2=0.3891
Epoch 004/800 | train_loss=624.3868 | train_r2=0.3907 | val_mae=746.9111 | val_rmse=2952.5787 | val_r2=0.3890
Epoch 005/800 | train_loss=624.2451 | train_r2=0.3906 | val_mae=746.9530 | val_rmse=2952.8747 | val_r2=0.3889
Epoch 006/800 | train_loss=624.2682 | train_r2=0.3905 | val_mae=747.0218 | val_rmse=2953.0662 | val_r2=0.3888
Epoch 007/800 | train_loss=624.4471 | train_r2=0.3909 | val_mae=746.9774 | val_rmse=2952.1956 | val_r2=0.3892
Epoch 008/800 | train_loss=624.2469 | train_r2=0.3906 | val_mae=747.0046 | val_rmse=2952.8830 | val_r2=0.3889
Epoch 009/800 | train_loss=624.3344 | train_r2=0.3908 | val_mae=746.8977 | val_rmse=2952.3630 | val_r2=0.3891
Epoch 010/

In [ ]:
# ------------ Learning curves for MAE and R-squared over epochs: ------------ #
import numpy as np
import matplotlib.pyplot as plt

# history columns:
# 0 train_loss, 1 train_mae, 2 train_mse, 3 train_rmse, 4 train_r2,
# 5 val_mae, 6 val_mse, 7 val_rmse, 8 val_r2
train_mae = [h[1] for h in history]
val_mae = [h[5] for h in history]
train_r2 = [h[4] for h in history]
val_r2 = [h[8] for h in history]
epochs = np.arange(1, len(history) + 1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

ax1.plot(epochs, train_mae, label='Train')
ax1.plot(epochs, val_mae, label='Validation')
ax1.set_ylabel('MAE')
ax1.legend()
ax1.grid(True)

ax2.plot(epochs, train_r2, label='Train')
ax2.plot(epochs, val_r2, label='Validation')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('R-squared')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

Original feature columns: ['CRF05', 'CRF07', 'CRF10', 'CRF11', 'CRF12', 'CRF13', 'CRF14', 'CRF15', 'CRF17A', 'CRF17B', 'CRF17C', 'CRF18', 'CRF19', 'CRF57', 'V106']
Categorical columns: ['CRF05', 'CRF10', 'CRF11', 'CRF12', 'CRF13', 'CRF14', 'CRF15', 'CRF17A', 'CRF17B', 'CRF17C', 'CRF18', 'CRF19', 'CRF57']
Numeric columns: ['CRF07', 'V106']
Categories for CRF05: [np.int64(0), np.int64(1)]
Categories for CRF10: [np.int64(0), np.int64(1)]
Categories for CRF11: [np.int64(0), np.int64(1)]
Categories for CRF12: [np.int64(0), np.int64(1)]
Categories for CRF13: [np.int64(0), np.int64(1)]
Categories for CRF14: [np.int64(0), np.int64(1)]
Categories for CRF15: [np.int64(0), np.int64(1)]
Categories for CRF17A: [np.int64(0), np.int64(1)]
Categories for CRF17B: [np.int64(0), np.int64(1)]
Categories for CRF17C: [np.int64(0), np.int64(1)]
Categories for CRF18: [np.int64(0), np.int64(1)]
Categories for CRF19: [np.int64(0), np.int64(1)]
Categories for CRF57: [np.int64(0), np.int64(1)]
Transformed feature

In [ ]:
# -------------------------------- Final Test -------------------------------- #
reloaded_model = CostRegressor(get_input_dim()).to(DEVICE)
reloaded_model.load_state_dict(torch.load(model_path, map_location=DEVICE))
test_mae, test_mse, test_rmse, test_r2 = evaluator(reloaded_model, test_loader)

print('Reloaded saved model evaluation on test split')
print(f'MAE:  {test_mae:.4f}')
print(f'MSE:  {test_mse:.4f}')
print(f'RMSE: {test_rmse:.4f}')
print(f'R2:   {test_r2:.4f}')

Reloaded saved model evaluation on test split
MAE:  558.8065
MSE:  2676774.1335
RMSE: 1636.0850
R2:   0.3649
